# Time Series

Time series data is ordered by time. The goal is usually to understand temporal patterns and forecast future values.

## Workflow

**Load → Parse Date → Sort by Time → Set Time Index → EDA → Check Missing/Irregular Dates → Resample/Transform → Train/Test Split by Time → Create Features/Lags → Train Model → Forecast → Evaluate → Compare → Improve**

**Important:** Never randomly shuffle a time series when creating a normal train/test split. The model must learn from the past and predict the future.

Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("time_series.csv")

df.head()

Parse Date and Set Time Index

In [ ]:
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date")
df = df.set_index("date")

print(df.head())
print(df.index.min(), "to", df.index.max())

EDA

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate timestamps:", df.index.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nSummary:")
display(df.describe())

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df.index, df["target"])
plt.title("Time Series")
plt.xlabel("Date")
plt.ylabel("Target")
plt.show()

## Time Series Patterns

Look for:

- **Trend** → long-term upward/downward movement
- **Seasonality** → repeating pattern at a fixed period
- **Cyclic pattern** → longer irregular rises/falls
- **Noise** → random variation
- **Outliers** → unusual observations

Also check whether the time interval is regular.

Check Frequency / Missing Dates

In [ ]:
print("Inferred frequency:", pd.infer_freq(df.index))

full_index = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq=pd.infer_freq(df.index)
)

missing_dates = full_index.difference(df.index)

print("Missing timestamps:", len(missing_dates))

## Preprocessing

Common time-series preprocessing:

- Convert date column to datetime
- Sort chronologically
- Set date as index
- Handle missing values
- Handle duplicate timestamps
- Resample when required
- Handle outliers when justified
- Transform the target when required
- Create lag/rolling features

**Do not use future information while creating features.**

Handle Missing Values

In [ ]:
df["target"] = df["target"].interpolate()

print(df["target"].isnull().sum())

Resampling

In [ ]:
# Example: convert daily data to monthly mean
monthly = df["target"].resample("ME").mean()

monthly.head()

Train/Test Split

In [ ]:
split_index = int(len(df) * 0.8)

train = df.iloc[:split_index].copy()
test = df.iloc[split_index:].copy()

print("Train:", train.shape)
print("Test:", test.shape)

print("Train end:", train.index.max())
print("Test start:", test.index.min())

# Baseline Forecast

A baseline gives us a simple model to beat.

**Naive forecast:** predict the next value using the previous observed value.

In [ ]:
y_train = train["target"]
y_test = test["target"]

naive_pred = pd.Series(
    y_train.iloc[-1],
    index=y_test.index
)

naive_pred.head()

Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_forecast(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return {
        "Model": name,
        "MAE": round(mae, 3),
        "RMSE": round(rmse, 3)
    }

baseline_result = evaluate_forecast(
    "Naive Baseline",
    y_test,
    naive_pred
)

baseline_result

## Time-Series Features

For machine-learning models, convert the time series into supervised learning using:

- Lag features
- Rolling mean
- Rolling standard deviation
- Calendar features
- Trend features

Example:

`target(t-1) → predict target(t)`

`target(t-7) → predict target(t)`

In [ ]:
data = df.copy()

data["lag_1"] = data["target"].shift(1)
data["lag_7"] = data["target"].shift(7)
data["lag_30"] = data["target"].shift(30)

data["rolling_mean_7"] = data["target"].shift(1).rolling(7).mean()
data["rolling_std_7"] = data["target"].shift(1).rolling(7).std()

data["day_of_week"] = data.index.dayofweek
data["month"] = data.index.month
data["year"] = data.index.year

data = data.dropna()

data.head()

Feature-Based Train/Test Split

In [ ]:
feature_cols = [
    "lag_1",
    "lag_7",
    "lag_30",
    "rolling_mean_7",
    "rolling_std_7",
    "day_of_week",
    "month",
    "year"
]

X = data[feature_cols]
y = data["target"]

split_index = int(len(data) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print(X_train.shape, X_test.shape)

# Linear Regression

**Use:** A simple baseline when lag/rolling features have an approximately linear relationship with the target.

In [ ]:
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()

model_lr.fit(X_train, y_train)

pred_lr = model_lr.predict(X_test)

result_lr = evaluate_forecast(
    "Linear Regression",
    y_test,
    pred_lr
)

result_lr

# Random Forest Regressor

**Use:** Non-linear relationships in lag/feature-based time-series forecasting.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)

result_rf = evaluate_forecast(
    "Random Forest",
    y_test,
    pred_rf
)

result_rf

# Gradient Boosting Regressor

**Use:** Strong tabular forecasting when engineered time-series features capture useful patterns.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

model_gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model_gb.fit(X_train, y_train)

pred_gb = model_gb.predict(X_test)

result_gb = evaluate_forecast(
    "Gradient Boosting",
    y_test,
    pred_gb
)

result_gb

# XGBoost Regressor

**Use:** High-performance machine-learning forecasting with lag, rolling and calendar features.

In [ ]:
# pip install xgboost

from xgboost import XGBRegressor

model_xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model_xgb.fit(X_train, y_train)

pred_xgb = model_xgb.predict(X_test)

result_xgb = evaluate_forecast(
    "XGBoost",
    y_test,
    pred_xgb
)

result_xgb

# ARIMA

**Use:** Classical statistical forecasting for a single time series where past values and differencing are useful.

`ARIMA(p,d,q)`

- **p** → autoregressive terms
- **d** → differencing
- **q** → moving-average terms

In [ ]:
# pip install statsmodels

from statsmodels.tsa.arima.model import ARIMA

arima_model = ARIMA(
    y_train,
    order=(1, 1, 1)
)

arima_fit = arima_model.fit()

pred_arima = arima_fit.forecast(
    steps=len(y_test)
)

result_arima = evaluate_forecast(
    "ARIMA",
    y_test,
    pred_arima
)

result_arima

# SARIMA

**Use:** ARIMA when the series also has a repeating seasonal pattern.

`SARIMA(p,d,q)(P,D,Q,s)`

`s` = seasonal period.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_model = SARIMAX(
    y_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12)
)

sarima_fit = sarima_model.fit(
    disp=False
)

pred_sarima = sarima_fit.forecast(
    steps=len(y_test)
)

result_sarima = evaluate_forecast(
    "SARIMA",
    y_test,
    pred_sarima
)

result_sarima

# Exponential Smoothing

**Use:** Forecasting data with level, trend and/or seasonality.

Common forms:
- Simple Exponential Smoothing
- Holt
- Holt-Winters / Exponential Smoothing

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model_hw = ExponentialSmoothing(
    y_train,
    trend="add",
    seasonal="add",
    seasonal_periods=12
)

fit_hw = model_hw.fit()

pred_hw = fit_hw.forecast(
    len(y_test)
)

result_hw = evaluate_forecast(
    "Holt-Winters",
    y_test,
    pred_hw
)

result_hw

# Model Comparison

**MAE → lower is better**

**RMSE → lower is better**

For time-series forecasting, compare models on a chronological validation/test period.

In [ ]:
results = pd.DataFrame([
    baseline_result,
    result_lr,
    result_rf,
    result_gb,
    result_xgb,
    result_arima,
    result_sarima,
    result_hw
])

results.sort_values("RMSE").reset_index(drop=True)

In [ ]:
comparison = results.sort_values("RMSE")

plt.figure(figsize=(10, 6))
plt.barh(comparison["Model"], comparison["RMSE"])
plt.xlabel("RMSE (lower is better)")
plt.title("Time-Series Model Comparison")
plt.gca().invert_yaxis()
plt.show()

# Model Improvement

**Baseline → Better Features → Transform/Stationarity → Compare Models → Time-Series Cross-Validation → Hyperparameter Tuning → Final Model → Forecast**

### Feature Engineering
- More useful lags
- Seasonal lags
- Rolling statistics
- Calendar features
- Trend features

### Transformations
- Log transformation
- Differencing
- Box-Cox transformation

### Seasonality
Identify the seasonal period and use seasonal models/features.

### Hyperparameter Tuning
Examples:
- ARIMA `(p,d,q)`
- SARIMA `(p,d,q)(P,D,Q,s)`
- Tree `n_estimators`, `max_depth`
- XGBoost `learning_rate`, `max_depth`, `n_estimators`

### Time-Series Cross-Validation

Do not use ordinary random `KFold` for forecasting.

Use expanding/rolling time-based validation so validation data always occurs after training data.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

tscv = TimeSeriesSplit(n_splits=5)

scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X), start=1):
    X_tr = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)

    pred = model.predict(X_val)

    rmse = np.sqrt(
        mean_squared_error(y_val, pred)
    )

    scores.append(rmse)

print("Fold RMSE:", scores)
print("Mean CV RMSE:", np.mean(scores))

# Forecast Visualization

Always inspect whether predictions follow the actual time-series pattern.

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(y_test.index, y_test, label="Actual")
plt.plot(y_test.index, pred_rf, label="Random Forest")

plt.title("Actual vs Forecast")
plt.xlabel("Date")
plt.ylabel("Target")
plt.legend()
plt.show()

# Quick Revision

**Naive** → previous value baseline

**Linear Regression** → lag/features have approximately linear relationship

**Random Forest** → non-linear feature-based forecasting

**Gradient Boosting** → strong tabular forecasting

**XGBoost** → powerful gradient boosting with engineered features

**ARIMA** → classical non-seasonal time-series model

**SARIMA** → ARIMA + seasonality

**Holt-Winters** → level + trend + seasonality

**TimeSeriesSplit** → chronological cross-validation

**MAE/RMSE** → lower is better

**Main rule:** In time series, the future must never leak into the past.